# Generate ChimeraX B-factor defattr files for HA binding

Reads `../../results/summaries/HA_binding.csv` and writes one ChimeraX `.defattr`
file per protein (alpha and beta).

Steps:
- Keep only rows where `entry in 292_M3_tat_H5 cells` is greater or equal than `-3`.
- Drop rows with a null/NA `structure_site`.
- Split into alpha (`structure_site` ending in `_a`) and beta (`_b`).
- Sum `HA binding escape` per site.
- Number each site by the numeric part of `structure_site` (e.g. `-1_a` -> `-1`, `2_a` -> `2`).


In [1]:
import pandas as pd

INPUT_CSV = "../../results/summaries/HA_binding.csv"
ENTRY_COL = "entry in 292_M3_tat_H5 cells"
ESCAPE_COL = "HA binding escape"
ENTRY_THRESHOLD = -3  # keep entries strictly greater than this


In [2]:
# Read input and apply filters
df = pd.read_csv(INPUT_CSV)
print(f"Read {len(df)} rows from {INPUT_CSV}")

# Keep only rows with entry strictly greater than -3 (NaN entries are dropped)
df = df[df[ENTRY_COL] >= ENTRY_THRESHOLD]
print(f"{len(df)} rows after filtering {ENTRY_COL} >= {ENTRY_THRESHOLD}")

# Drop rows where structure_site is null/NA -- not of interest
df = df[df["structure_site"].notna()].copy()
print(f"{len(df)} rows after dropping null structure_site")

# Numeric site number (strip the _a / _b protein suffix)
df["structure_site"] = df["structure_site"].astype(str)
df["site_number"] = df["structure_site"].str.replace(r"_[ab]$", "", regex=True).astype(int)

# Protein label from the suffix
df["protein"] = df["structure_site"].str.extract(r"_([ab])$")[0].map({"a": "alpha", "b": "beta"})
print(df["protein"].value_counts(dropna=False).to_string())


Read 8140 rows from ../../results/summaries/HA_binding.csv
7496 rows after filtering entry in 292_M3_tat_H5 cells >= -3
7384 rows after dropping null structure_site
protein
beta     3696
alpha    3688


In [3]:
def write_defattr(protein_df, output_file):
    """Sum HA binding escape per site and write a ChimeraX defattr file."""
    site_sums = (
        protein_df.groupby("site_number")[ESCAPE_COL]
        .sum()
        .reset_index()
        .sort_values("site_number")
    )

    with open(output_file, "w") as f:
        f.write("# ChimeraX defattr file\n")
        f.write("#\n")
        f.write("attribute: binding\n")
        f.write("match mode: any\n")
        f.write("recipient: residues\n")
        f.write("\n")
        for _, row in site_sums.iterrows():
            f.write(f"\t:{int(row['site_number'])}\t{float(row[ESCAPE_COL]):.6f}\n")

    print(f"Wrote {output_file}: {len(site_sums)} sites")
    print(f"  binding escape  min={site_sums[ESCAPE_COL].min():.6f}  "
          f"max={site_sums[ESCAPE_COL].max():.6f}  "
          f"mean={site_sums[ESCAPE_COL].mean():.6f}")
    print("  example lines:")
    print(site_sums.head(5).to_string(index=False))
    return site_sums


In [4]:
# Write one defattr file per protein
for protein, output_file in [
    ("alpha", "bfactors_binding_alpha.defattr"),
    ("beta", "bfactors_binding_beta.defattr"),
]:
    print(f"\n=== {protein} ===")
    write_defattr(df[df["protein"] == protein], output_file)



=== alpha ===
Wrote bfactors_binding_alpha.defattr: 196 sites
  binding escape  min=-13.269600  max=15.671100  mean=-0.677276
  example lines:
 site_number  HA binding escape
          -1          -0.341960
           1          -0.317202
           2           0.587940
           3           0.036444
           4          -1.151197

=== beta ===
Wrote bfactors_binding_beta.defattr: 198 sites
  binding escape  min=-6.610440  max=6.128225  mean=-0.612136
  example lines:
 site_number  HA binding escape
           2          -1.710456
           3          -0.095228
           4          -0.561300
           5           0.257992
           6          -0.289165
